In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from db_config import get_engine

engine = get_engine()
# Fetching from the newly created table
df = pd.read_sql("SELECT * FROM pharma_supply_clean", con=engine)

# ABC Classification by Optimal Stock Level (proxy for inventory value)
df['annual_restock_freq'] = pd.to_numeric(df['annual_restock_freq'], errors='coerce')

df_abc = df.groupby('Drug').agg(
    total_value=('Optimal_Stock_Level', 'sum'),
    avg_demand=('Demand_Forecast', 'mean'),
    avg_restock_freq=('annual_restock_freq', 'mean'),
    is_understocked=('is_understocked', 'max')
).reset_index()

df_abc = df_abc.sort_values('total_value', ascending=False)
df_abc['cumulative_pct'] = (
    df_abc['total_value'].cumsum()
    / df_abc['total_value'].sum() * 100
)

def classify_abc(pct):
    if pct <= 70:   return 'A'  # Top 70% of stock allocation
    elif pct <= 90: return 'B'  # Next 20%
    else:           return 'C'  # Bottom 10%

df_abc['abc_class'] = df_abc['cumulative_pct'].apply(classify_abc)

print(df_abc['abc_class'].value_counts())
print("\nOptimal Stock Volume by class:")
print(df_abc.groupby('abc_class')['total_value'].sum())

abc_class
A    2
B    1
C    1
Name: count, dtype: int64

Optimal Stock Volume by class:
abc_class
A    315207975
B    155940312
C    155867245
Name: total_value, dtype: int64


In [3]:
# Save ABC results to MySQL
df_abc.to_sql('pharma_abc_classification', con=engine,
              if_exists='replace', index=False)
print("ABC table saved to MySQL")

# Verify in SQL (run in VS Code .sql file):
# SELECT abc_class, COUNT(*) as drugs,
#        ROUND(SUM(total_value),2) as total_optimal_stock
# FROM pharma_abc_classification GROUP BY abc_class;

ABC table saved to MySQL


In [4]:
import numpy as np

# Safety stock formula from industrial/operations engineering:
# Safety Stock = Z × σ_demand × √(lead_time)
# ROP = (avg_daily_demand × lead_time) + safety_stock

Z_scores = {'A': 2.33, 'B': 1.65, 'C': 1.28}
# A-class: 99% service level (critical drugs)
# B-class: 95% service level
# C-class: 90% service level

results = []
for abc_class, group in df.groupby(
    df['Drug'].map(df_abc.set_index('Drug')['abc_class'])):

    # Approximate daily demand from the total forecast
    avg_demand   = (group['Demand_Forecast'] / 365).mean()
    std_demand   = (group['Demand_Forecast'] / 365).std() if len(group) > 1 else avg_demand * 0.2
    
    # Approximate lead time in days (365 days / restock frequency)
    lead_time    = (365 / group['annual_restock_freq']).mean()
    Z            = Z_scores.get(abc_class, 1.65)

    safety_stock = Z * std_demand * np.sqrt(lead_time)
    rop          = (avg_demand * lead_time) + safety_stock

    results.append({
        'abc_class':    abc_class,
        'avg_daily_demand': round(avg_demand, 2),
        'avg_lead_time_days': round(lead_time, 1),
        'safety_stock': round(safety_stock, 0),
        'recommended_rop': round(rop, 0)
    })

rop_df = pd.DataFrame(results)
print(rop_df)
rop_df.to_sql('pharma_rop_recommendations', con=engine,
              if_exists='replace', index=False)
print("Saved ROP recommendations to MySQL")

  abc_class  avg_daily_demand  avg_lead_time_days  safety_stock  \
0         A             15.06                43.0         109.0   
1         B             15.03                43.1          77.0   
2         C             15.01                42.8          59.0   

   recommended_rop  
0            757.0  
1            725.0  
2            701.0  
Saved ROP recommendations to MySQL
